# 02 Polynomial Interpolation

This notebook covers the main global polynomial tools in Chapter 2:

* Lagrange basis polynomials;
* Newton divided differences;
* Runge behavior on equally spaced nodes;
* Chebyshev nodes as a practical correction.


In [ ]:
import pathlib
import sys

import matplotlib.pyplot as plt
import numpy as np

ROOT = pathlib.Path.cwd()
while not (ROOT / "src" / "py_sc").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from py_sc import (
    chebyshev_nodes,
    divided_difference_table,
    lagrange_interpolate,
    newton_divided_differences,
    newton_interpolate,
)


## Lagrange basis functions

The Lagrange interpolant is

$$
p_n(x)=\sum_{j=0}^{n}y_jL_j(x),\qquad
L_j(x)=\prod_{m\ne j}\frac{x-x_m}{x_j-x_m}.
$$

Each basis function satisfies $L_j(x_i)=\delta_{ij}$, so it selects one data
value and vanishes at the other nodes.


In [ ]:
def lagrange_basis(nodes, basis_index, x_eval):
    values = np.ones_like(x_eval, dtype=float)
    xj = nodes[basis_index]
    for m, xm in enumerate(nodes):
        if m != basis_index:
            values *= (x_eval - xm) / (xj - xm)
    return values

nodes = np.array([-1.0, -0.3, 0.4, 1.0])
x_plot = np.linspace(-1.1, 1.1, 500)

plt.figure(figsize=(8, 4.5))
for j in range(nodes.size):
    plt.plot(x_plot, lagrange_basis(nodes, j, x_plot), label=f"L_{j}")
plt.scatter(nodes, np.zeros_like(nodes), color="black", zorder=3)
plt.xlabel("x")
plt.ylabel("basis value")
plt.title("Lagrange basis polynomials")
plt.legend()
plt.tight_layout()
plt.show()


## Newton divided differences

The Newton form is

$$
p_n(x)=c_0+c_1(x-x_0)+c_2(x-x_0)(x-x_1)+\cdots.
$$

The coefficients are the first row of the divided-difference table. This form is
especially useful when adding new nodes because the old coefficients do not need
to be recomputed from scratch.


In [ ]:
x = np.array([0.0, 1.0, 2.0, 3.0])
y = np.array([1.0, 2.0, 0.0, 4.0])
x_eval = np.linspace(0.0, 3.0, 300)

lagrange_values = lagrange_interpolate(x, y, x_eval)
newton_nodes, coefficients = newton_divided_differences(x, y)
newton_values = newton_interpolate(newton_nodes, coefficients, x_eval)
_, table = divided_difference_table(x, y)

print("Newton coefficients:", coefficients)
print("First row of divided-difference table:", table[0])

plt.figure(figsize=(8, 4.5))
plt.plot(x_eval, lagrange_values, label="Lagrange form")
plt.plot(x_eval, newton_values, "--", label="Newton form")
plt.scatter(x, y, color="black", zorder=3, label="Data")
plt.xlabel("x")
plt.ylabel("p(x)")
plt.title("Lagrange and Newton forms represent the same polynomial")
plt.legend()
plt.tight_layout()
plt.show()


## Runge behavior

The interpolation error formula contains the nodal product

$$
\prod_{i=0}^{n}(x-x_i).
$$

For high-degree interpolation on equally spaced nodes, this product can become
large near the endpoints. The Runge function is a standard stress test.


In [ ]:
def runge(x):
    return 1.0 / (1.0 + 25.0 * x**2)

x_plot = np.linspace(-1.0, 1.0, 1000)
y_true = runge(x_plot)

fig, axes = plt.subplots(1, 2, figsize=(11, 4), sharey=True)
for ax, n in zip(axes, [9, 17]):
    x_equal = np.linspace(-1.0, 1.0, n)
    y_equal = runge(x_equal)
    y_interp = lagrange_interpolate(x_equal, y_equal, x_plot)
    ax.plot(x_plot, y_true, color="black", label="Runge function")
    ax.plot(x_plot, y_interp, label=f"Equal nodes, n={n}")
    ax.scatter(x_equal, y_equal, s=18, zorder=3)
    ax.set_xlabel("x")
    ax.set_title(f"Degree {n - 1}")
axes[0].set_ylabel("y")
axes[0].legend()
fig.suptitle("Endpoint oscillation with equally spaced nodes")
fig.tight_layout()
plt.show()


## Chebyshev nodes

Chebyshev nodes cluster near the endpoints and reduce the growth of the nodal
product. They do not make global polynomial interpolation perfect, but they are
a central practical tool.


In [ ]:
node_counts = [9, 17, 25]
for n in node_counts:
    equal_nodes = np.linspace(-1.0, 1.0, n)
    cheb_nodes = chebyshev_nodes(-1.0, 1.0, n)

    equal_error = np.max(np.abs(lagrange_interpolate(equal_nodes, runge(equal_nodes), x_plot) - y_true))
    cheb_error = np.max(np.abs(lagrange_interpolate(cheb_nodes, runge(cheb_nodes), x_plot) - y_true))
    print(f"nodes={n:2d}, equal error={equal_error:10.3e}, chebyshev error={cheb_error:10.3e}")

n = 17
x_equal = np.linspace(-1.0, 1.0, n)
x_cheb = chebyshev_nodes(-1.0, 1.0, n)

plt.figure(figsize=(8, 4.5))
plt.plot(x_plot, y_true, color="black", label="Runge function")
plt.plot(x_plot, lagrange_interpolate(x_equal, runge(x_equal), x_plot), label="Equal nodes")
plt.plot(x_plot, lagrange_interpolate(x_cheb, runge(x_cheb), x_plot), label="Chebyshev nodes")
plt.scatter(x_equal, runge(x_equal), s=18, alpha=0.7)
plt.scatter(x_cheb, runge(x_cheb), s=18, alpha=0.7)
plt.xlabel("x")
plt.ylabel("y")
plt.title("Chebyshev nodes reduce endpoint oscillation")
plt.legend()
plt.tight_layout()
plt.show()


## Summary

* Lagrange form is direct and basis-oriented.
* Newton form is incremental and table-oriented.
* High-degree interpolation on equally spaced nodes can fail badly.
* Chebyshev nodes are an important remedy for smooth functions on a bounded interval.
